# 🧠❤️ EMOTIONAI — EMOTION PREDICTION

**PROJECT OVERVIEW**

This project analyzes text data to identify emotions expressed in written sentences and prepare the data for Natural Language Processing (NLP) and machine learning.

The project includes Exploratory Data Analysis (EDA), Text Data Cleaning & Preprocessing, Feature Extraction using Bag-of-Words (BoW), Label Encoding, and Model Training to classify different emotions from text.

Multiple classification models were trained and compared, including Logistic Regression and Naive Bayes.

The models were evaluated using Accuracy Score, F1-Score, Confusion Matrix, and Classification Report to compare their performance.

The best-performing model was selected based on the overall classification performance and used for real-time emotion prediction.

**Models Used:** Logistic Regression, Naive Bayes

**NLP Technique:** Bag-of-Words (BoW)

**Evaluation:** Accuracy Score, F1-Score, Confusion Matrix, Classification Report

**Best Model:** Selected based on the best overall Accuracy and F1-Score.

**Deployment:** FastAPI backend with a web frontend for real-time emotion prediction.


**Import Libraries**

In [238]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

**Load data**

In [239]:
df = pd.read_csv('sentiments_data.txt', sep=';', header=None, names=['text','emotions'])
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


# Data Cleaning

## Check null values

In [240]:
df.isnull().sum()

,0
text,0
emotions,0


## Encoding  by using labelEncoder

In [241]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['emotions'] = le.fit_transform(df['emotions'])

In [242]:
df

,text,emotions
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,0
...,...,...
16028,i feel close to the person i love,3
16029,my partner is very dear to me,3
16030,i have a special connection with her,3
16031,i feel a strong attachment to my partner,3


## Convert text in lowercase

In [243]:
df['text'] = df['text'].str.lower()

## Remove Punctuation

In [244]:
df['text'] = df['text'].str.replace(r'[^\w\s]', '', regex=True)

## Remove Number

In [245]:
df['text'] = df['text'].str.replace(r'\d+', '', regex=True)

## Remove URLs/Links

In [246]:
df['text'] = df['text'].str.replace(r'https?://\S+|www\.\S+', '', regex=True)

## Remove HTML Tags

In [247]:
df['text'] = df['text'].str.replace(r'<.*?>', '', regex=True)

## Remove Emoji's & Special Characters

In [248]:
df['text'] = df['text'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

## Remove STOP words

In [249]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

**before**

In [250]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

stop_words -= {'not', 'no', 'never'}

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [251]:
def remove(txt):
    return ' '.join(word for word in txt.split() if word not in stop_words)

df['text'] = df['text'].apply(remove)

In [252]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

**After**

Removed common/less meaningful words from the text using NLTK stop words. Kept the important words that carry more meaning for NLP analysis and reduced unnecessary words.

# Feature Extraction / vectorization


## 1. Import Libraries

In [253]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 2. Split Data into Training and Testing Sets


In [254]:
from sklearn.model_selection import train_test_split

# input feature
X = df['text']

# target variable
y = df['emotions']

# split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## 1. Bag of Words

In [255]:
# create BoW vectorizer
bow = CountVectorizer(ngram_range=(1, 2))

# fit only on training data
X_train_bow = bow.fit_transform(X_train)

# transform test data
X_test_bow = bow.transform(X_test)

print("BoW training shape:", X_train_bow.shape)
print("BoW testing shape:", X_test_bow.shape)

BoW training shape: (12826, 92334)
BoW testing shape: (3207, 92334)


## 2. TD-IDF

In [256]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000
)

# fit only on training data
X_train_tfidf = tfidf.fit_transform(X_train)

# transform test data
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)

TF-IDF training shape: (12826, 10000)
TF-IDF testing shape: (3207, 10000)


**Check**

In [257]:
print("BoW:", X_train_bow.shape)
print("TF-IDF:", X_train_tfidf.shape)

BoW: (12826, 92334)
TF-IDF: (12826, 10000)


12,800 → number of training text samples.

13,377 → number of unique words/features in the vocabulary.

BoW and TF-IDF have the same shape because both use the same text data and vocabulary size.

BoW stores word counts, while TF-IDF stores word importance scores.

In short:

(12,800, 13,377) = 12,800 text samples × 13,377 text features.

# Model Creation

we use Multinomial Navi bayes and logistis regression model. Compare their Accuracy, Precision, Recall and F1-score and choose the best-performing combination for your emotion classification project.

## 1. Import Libraries

In [258]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## 2. Multinomial Naive Bayes — BoW

In [259]:
# create model
nb_bow = MultinomialNB()

# train model
nb_bow.fit(X_train_bow, y_train)

# make predictions
y_pred_nb_bow = nb_bow.predict(X_test_bow)

# evaluate
print("Multinomial NB + BoW")
print("Accuracy:", accuracy_score(y_test, y_pred_nb_bow))
print(classification_report(y_test, y_pred_nb_bow))

Multinomial NB + BoW
Accuracy: 0.7405675085749922
              precision    recall  f1-score   support

           0       0.95      0.48      0.64       432
           1       0.88      0.51      0.65       388
           2       0.69      0.97      0.80      1073
           3       0.89      0.15      0.26       267
           4       0.73      0.95      0.83       933
           5       1.00      0.04      0.08       114

    accuracy                           0.74      3207
   macro avg       0.86      0.52      0.54      3207
weighted avg       0.79      0.74      0.70      3207



## 3. Multinomial Naive Bayes — TF-IDF

In [260]:
# create model
nb_tfidf = MultinomialNB()

# train model
nb_tfidf.fit(X_train_tfidf, y_train)

# make predictions
y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

# evaluate
print("Multinomial NB + TF-IDF")
print("Accuracy:", accuracy_score(y_test, y_pred_nb_tfidf))
print(classification_report(y_test, y_pred_nb_tfidf))

Multinomial NB + TF-IDF
Accuracy: 0.7380729653882133
              precision    recall  f1-score   support

           0       0.95      0.47      0.63       432
           1       0.90      0.49      0.64       388
           2       0.67      0.98      0.79      1073
           3       0.98      0.15      0.26       267
           4       0.76      0.94      0.84       933
           5       1.00      0.04      0.07       114

    accuracy                           0.74      3207
   macro avg       0.87      0.51      0.54      3207
weighted avg       0.80      0.74      0.70      3207



## 4. Logistic Regression — BoW

In [261]:
# create model
lr_bow = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

# train model
lr_bow.fit(X_train_bow, y_train)

# make predictions
y_pred_lr_bow = lr_bow.predict(X_test_bow)

# evaluate
print("Logistic Regression + BoW")
print("Accuracy:", accuracy_score(y_test, y_pred_lr_bow))
print(classification_report(y_test, y_pred_lr_bow))

Logistic Regression + BoW
Accuracy: 0.899282818833801
              precision    recall  f1-score   support

           0       0.90      0.87      0.88       432
           1       0.90      0.87      0.88       388
           2       0.92      0.91      0.91      1073
           3       0.75      0.85      0.80       267
           4       0.95      0.93      0.94       933
           5       0.75      0.86      0.80       114

    accuracy                           0.90      3207
   macro avg       0.86      0.88      0.87      3207
weighted avg       0.90      0.90      0.90      3207



## 5. Logistic Regression — TF-IDF

In [262]:
# create model
lr_tfidf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

# train model
lr_tfidf.fit(X_train_tfidf, y_train)

# make predictions
y_pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

# evaluate
print("Logistic Regression + TF-IDF")
print("Accuracy:", accuracy_score(y_test, y_pred_lr_tfidf))
print(classification_report(y_test, y_pred_lr_tfidf))

Logistic Regression + TF-IDF
Accuracy: 0.8930464608668538
              precision    recall  f1-score   support

           0       0.88      0.88      0.88       432
           1       0.89      0.86      0.87       388
           2       0.94      0.89      0.91      1073
           3       0.74      0.92      0.82       267
           4       0.95      0.91      0.93       933
           5       0.66      0.91      0.77       114

    accuracy                           0.89      3207
   macro avg       0.84      0.89      0.86      3207
weighted avg       0.90      0.89      0.90      3207



**Use Logistic Regression + BoW because:**

It has the highest accuracy: **89%.**
It has the highest weighted F1-score: **90%.**
It performs well across all 6 emotion classes.
Compared with Naive Bayes, it has much better recall and F1-score, especially for less frequent emotions like love and surprise.
TF-IDF performed slightly worse than BoW for this particular dataset.

**Logistic Regression + BoW** was selected as the best model because it achieved the highest accuracy (89%) and F1-score (90%), providing the best overall performance for emotion classification.

# Save the Model

In [263]:
import joblib

# save the trained model
joblib.dump(lr_bow, 'emotion_logistic_regression.pkl')

# save the BoW vectorizer
joblib.dump(bow, 'emotion_bow_vectorizer.pkl')

# save the label encoder
joblib.dump(le, 'emotion_label_encoder.pkl')

print("Model, vectorizer, and label encoder saved successfully!")

Model, vectorizer, and label encoder saved successfully!
